## Example 1: CORS(app) answers a cross-origin request

In [ ]:
import importlib.metadata
from flask import Flask
from flask_cors import CORS
import flask_cors

print("flask_cors", flask_cors.__version__)
assert flask_cors.__version__ == importlib.metadata.version("flask-cors")

app = Flask(__name__)
CORS(app)  # defaults: every route, every origin


@app.route("/hello")
def hello():
    return {"message": "hi"}


client = app.test_client()
cross = client.get("/hello", headers={"Origin": "https://site.example"})
assert cross.status_code == 200 and cross.get_json() == {"message": "hi"}
# By default the caller's origin is echoed back (send_wildcard=False), with "Vary: Origin" for caches.
print("Allow-Origin:", cross.headers.get("Access-Control-Allow-Origin"), "| Vary:", cross.headers.get("Vary"))
assert cross.headers["Access-Control-Allow-Origin"] == "https://site.example", dict(cross.headers)
assert "Origin" in cross.headers["Vary"], cross.headers["Vary"]

wildcard_app = Flask("wildcard")
CORS(wildcard_app, send_wildcard=True)
wildcard_app.add_url_rule("/hello", view_func=hello)
wildcard = wildcard_app.test_client().get("/hello", headers={"Origin": "https://site.example"})
print("with send_wildcard=True:", wildcard.headers.get("Access-Control-Allow-Origin"))
assert wildcard.headers["Access-Control-Allow-Origin"] == "*", dict(wildcard.headers)
print("OK -- CORS(app) allows a cross-origin GET from any origin, echoed or as *")

## Example 2: A preflight request: allowed methods, headers and max age

In [ ]:
from flask import Flask
from flask_cors import CORS

app = Flask(__name__)
CORS(app, methods=["GET", "PUT"], allow_headers=["Content-Type", "X-Token"], max_age=600)


@app.route("/items/<int:item_id>", methods=["GET", "PUT"])
def item(item_id):
    return {"id": item_id}


client = app.test_client()
preflight = client.options("/items/7", headers={
    "Origin": "https://site.example",
    "Access-Control-Request-Method": "PUT",
    "Access-Control-Request-Headers": "X-Token",
})
headers = preflight.headers
print("preflight status:", preflight.status_code)
for name in ("Access-Control-Allow-Origin", "Access-Control-Allow-Methods",
             "Access-Control-Allow-Headers", "Access-Control-Max-Age"):
    print(f"  {name}: {headers.get(name)}")
assert preflight.status_code == 200
assert set(headers["Access-Control-Allow-Methods"].split(", ")) == {"GET", "PUT"}, headers["Access-Control-Allow-Methods"]
assert headers["Access-Control-Allow-Headers"].lower() == "x-token", headers["Access-Control-Allow-Headers"]
assert headers["Access-Control-Max-Age"] == "600"

# A header outside allow_headers is simply not echoed back, so the browser would block the real request.
refused = client.options("/items/7", headers={
    "Origin": "https://site.example",
    "Access-Control-Request-Method": "PUT",
    "Access-Control-Request-Headers": "X-Secret",
})
assert "Access-Control-Allow-Headers" not in refused.headers, dict(refused.headers)
print("OK -- the preflight allows PUT with X-Token for 600 s, and does not allow X-Secret")

## Example 3: Per-path resources and an origin allow-list

In [ ]:
from flask import Flask
from flask_cors import CORS

app = Flask(__name__)
CORS(app, resources={r"/api/*": {"origins": ["https://allowed.example"]}})


@app.route("/api/data")
def data():
    return {"rows": [1, 2, 3]}


@app.route("/private")
def private():
    return {"secret": True}


client = app.test_client()
allowed = client.get("/api/data", headers={"Origin": "https://allowed.example"})
other = client.get("/api/data", headers={"Origin": "https://other.example"})
outside = client.get("/private", headers={"Origin": "https://allowed.example"})
for label, response in (("allowed origin", allowed), ("other origin", other), ("path outside /api", outside)):
    print(f"{label:18} -> {response.status_code}, Allow-Origin: {response.headers.get('Access-Control-Allow-Origin')}")
assert allowed.headers["Access-Control-Allow-Origin"] == "https://allowed.example"  # the listed origin, not "*"
# The server still answers the others; only the CORS header is missing, so a browser hides the body.
assert other.status_code == 200 and "Access-Control-Allow-Origin" not in other.headers
assert outside.status_code == 200 and "Access-Control-Allow-Origin" not in outside.headers
print("OK -- only /api/* and only https://allowed.example get a CORS header")

## Example 4: @cross_origin on one route, with credentials and exposed headers

In [ ]:
from flask import Flask
from flask_cors import cross_origin

app = Flask(__name__)


@app.route("/account")
@cross_origin(origins=["https://app.example"], supports_credentials=True, expose_headers=["X-Request-Id"])
def account():
    return {"user": "alice"}, 200, {"X-Request-Id": "req-42"}


@app.route("/plain")
def plain():
    return {"cors": False}


client = app.test_client()
response = client.get("/account", headers={"Origin": "https://app.example"})
print("Allow-Origin:", response.headers.get("Access-Control-Allow-Origin"))
print("Allow-Credentials:", response.headers.get("Access-Control-Allow-Credentials"))
print("Expose-Headers:", response.headers.get("Access-Control-Expose-Headers"))
assert response.get_json() == {"user": "alice"} and response.headers["X-Request-Id"] == "req-42"
assert response.headers["Access-Control-Allow-Origin"] == "https://app.example"
assert response.headers["Access-Control-Allow-Credentials"] == "true"
assert response.headers["Access-Control-Expose-Headers"] == "X-Request-Id"
# The decorator applies to its own route only.
plain_response = client.get("/plain", headers={"Origin": "https://app.example"})
assert "Access-Control-Allow-Origin" not in plain_response.headers, dict(plain_response.headers)
print("OK -- @cross_origin allows credentials and exposes X-Request-Id on /account only")

## Example 5: Origins given as a regular expression

In [ ]:
from flask import Flask
from flask_cors import CORS

app = Flask(__name__)
# Origins can be regular expressions: here any HTTPS subdomain of example.com.
CORS(app, origins=[r"https://[a-z0-9-]+\.example\.com"])


@app.route("/ping")
def ping():
    return "pong"


client = app.test_client()
results = {}
for origin in ("https://shop.example.com", "https://api-2.example.com",
               "http://shop.example.com", "https://example.com.evil.test"):
    response = client.get("/ping", headers={"Origin": origin})
    results[origin] = response.headers.get("Access-Control-Allow-Origin")
    print(f"{origin:32} -> {results[origin]}")
assert results["https://shop.example.com"] == "https://shop.example.com"
assert results["https://api-2.example.com"] == "https://api-2.example.com"
assert results["http://shop.example.com"] is None  # plain http does not match the pattern
assert results["https://example.com.evil.test"] is None  # the whole origin must match, not a prefix
print("OK -- the origin pattern matches the two HTTPS subdomains and nothing else")